# DRL Product Recommendation — Train / Validation / Test Colab

Notebook này dùng pipeline đúng hơn:

1. Preprocess raw CSV thành `indexed_history.pkl`.
2. Split theo thời gian thành:
   - `train_history.pkl`
   - `val_history.pkl`
   - `test_history.pkl`
3. Train các model trên **train**.
4. Chọn model tốt nhất bằng **validation HitRate@5**.
5. Evaluate model đã chọn một lần cuối trên **test**.
6. So sánh test với Random / Popularity / Recent-item baseline.

Raw CSV cần nằm ở:

`MyDrive/drl-product-recommendation/colab_inputs/`

Cần đủ 4 file:

- `orders.csv`
- `products.csv`
- `order_products__prior.csv`
- `order_products__train.csv`

## 1. Mount Drive và cấu hình

In [ ]:
import os
import sys
from pathlib import Path

# Tự động phát hiện môi trường Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Định nghĩa hàm display fallback nếu chạy ngoài môi trường Jupyter/IPython
if "display" not in globals():
    try:
        from IPython.display import display
    except ImportError:
        def display(x):
            print(x)

REPO_URL = "https://github.com/dangnguyenhoai/drl-product-recommendation.git"
BRANCH = "main"

STATE_SIZE = 5
TOP_K = 5
TOP_N_ITEMS = 1000

# Tham số số tập chạy thử/thật
TRAIN_EPISODES = 5000
EVAL_EPISODES = 1000

# Tự động chọn device nếu có GPU
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if IN_COLAB:
    print("Môi trường: Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    
    PROJECT_DIR = Path("/content/drl-product-recommendation")
    RAW_DIR = Path("/content/drive/MyDrive/drl-product-recommendation/colab_inputs")
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/drl-product-recommendation/colab_outputs")
    N_USERS = None  # Dùng toàn bộ user trên Colab
else:
    print("Môi trường: Local")
    PROJECT_DIR = Path(os.getcwd())
    RAW_DIR = PROJECT_DIR / "data"
    DRIVE_OUTPUT_DIR = PROJECT_DIR / "outputs"
    # Local chạy nhẹ hơn để test
    N_USERS = 100
    TRAIN_EPISODES = 10  # Chạy thử nghiệm ít episode ở local để tránh lâu
    EVAL_EPISODES = 5

# Đảm bảo đường dẫn dự án có trong sys.path để import các module local
sys.path.insert(0, str(PROJECT_DIR.absolute()))

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DIR:", RAW_DIR)
print("DRIVE_OUTPUT_DIR:", DRIVE_OUTPUT_DIR)
print("N_USERS:", N_USERS)
print("TRAIN_EPISODES:", TRAIN_EPISODES)
print("EVAL_EPISODES:", EVAL_EPISODES)
print("DEVICE:", DEVICE)

## 2. Kiểm tra GPU

In [ ]:
!nvidia-smi

Wed Jun  3 02:27:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Clone repo và cài thư viện

Cell này luôn quay về `/content` trước khi clone để tránh lỗi `getcwd() failed`.

In [ ]:
import shutil
import subprocess
from pathlib import Path
import os

def run(cmd, cwd=None, capture=True):
    print("\n>>>", cmd)
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=capture,
        cwd=cwd,
    )

    if capture:
        if result.stdout:
            print("\nSTDOUT:")
            print(result.stdout)
        if result.stderr:
            print("\nSTDERR:")
            print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}: {cmd}")

    return result.stdout if capture else ""

if IN_COLAB:
    os.chdir("/content")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run(f"git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}", cwd="/content")
    os.chdir(PROJECT_DIR)
    
    # Cài thư viện phụ trên Colab
    run("pip install -q numpy pandas matplotlib ipython tabulate")
else:
    print("Đang chạy ở local. Bỏ qua bước clone git và cài đặt pip.")

print("Thư mục hiện tại:", os.getcwd())
run("python --version")
run('python -c "import torch; print(\'torch:\', torch.__version__); print(\'cuda:\', torch.cuda.is_available())"')

## 4. Sanity check repo

Nếu cell này lỗi, repo `main` chưa đúng code. Đừng train.

In [ ]:
import sys
import subprocess
from pathlib import Path

# Kiểm tra sự tồn tại của file bằng Python để chạy được cả trên Windows và Linux
required_files = [
    "utils/split_history_train_val_test.py",
    "evaluation/run_val_test_suite.py",
    "evaluation/metrics.py",
    "visualization/visualization.py"
]
for f_path in required_files:
    assert Path(f_path).exists(), f"Không tìm thấy file: {f_path}"
print("Mọi file quan trọng đều tồn tại.")

# Kiểm tra tham số CLI bằng Python (thay thế grep/findstr)
scripts = ["training.train_dqn", "evaluation.evaluate_dqn", "evaluation.inspect_policy"]
for script in scripts:
    res = subprocess.run(
        [sys.executable, "-m", script, "--help"],
        capture_output=True,
        text=True
    )
    assert "recent_boost" in res.stdout or "recent_boost" in res.stderr, f"{script} thiếu tham số --recent_boost"
print("Tất cả script CLI đều hỗ trợ --recent_boost.")

# Compile nhanh các file nguồn
import py_compile
files_to_compile = [
    "env/recommendation_env.py",
    "models/dqn_model.py",
    "models/dqn_agent.py",
    "training/train_dqn.py",
    "evaluation/evaluate_dqn.py",
    "evaluation/evaluate_baselines.py",
    "evaluation/evaluate_recent_baseline.py",
    "evaluation/metrics.py",
    "evaluation/inspect_policy.py",
    "evaluation/run_val_test_suite.py",
    "visualization/visualization.py",
    "utils/inspect_data.py",
    "utils/split_history_train_val_test.py"
]
for f in files_to_compile:
    py_compile.compile(f)
print("Compile tất cả file nguồn thành công.")

# Kiểm tra model
from models.dqn_model import DQN
_ = DQN(
    state_dim=5,
    action_dim=1000,
    embedding_dim=32,
    hidden_dim=128,
)
print("Khởi tạo DQN model (GRUDuelingDQN) thành công.")

# Kiểm tra Loss
agent_text = Path("models/dqn_agent.py").read_text(encoding="utf-8")
if "SmoothL1Loss" in agent_text:
    print("DQN loss: SmoothL1Loss OK")
else:
    print("WARNING: SmoothL1Loss not found. Check models/dqn_agent.py")

print("Sanity Check: OK")

## 5. Preprocess raw CSV → `indexed_history.pkl`

Không cần upload sẵn `indexed_history.pkl`. Notebook sẽ tạo lại từ raw CSV trong Drive.

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

required_files = {
    "orders": RAW_DIR / "orders.csv",
    "products": RAW_DIR / "products.csv",
    "prior": RAW_DIR / "order_products__prior.csv",
    "train": RAW_DIR / "order_products__train.csv",
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Không thấy file {name}: {path}")

print("Found raw files:")
for name, path in required_files.items():
    print(f"- {name}: {path}")

orders = pd.read_csv(required_files["orders"])
products = pd.read_csv(required_files["products"])
prior = pd.read_csv(required_files["prior"])
train = pd.read_csv(required_files["train"])

print("\nRaw shapes:")
print("orders:", orders.shape)
print("products:", products.shape)
print("prior:", prior.shape)
print("train:", train.shape)

order_products = pd.concat([prior, train], ignore_index=True)

merged = order_products.merge(
    orders[["order_id", "user_id", "order_number"]],
    on="order_id",
    how="inner",
)

sort_cols = ["user_id", "order_number"]
if "add_to_cart_order" in merged.columns:
    sort_cols.append("add_to_cart_order")

merged = merged.sort_values(sort_cols)

top_items = (
    merged["product_id"]
    .value_counts()
    .head(TOP_N_ITEMS)
    .index
    .tolist()
)

item_to_index = {product_id: idx for idx, product_id in enumerate(top_items)}

all_users = sorted(merged["user_id"].unique())
selected_users = all_users[:N_USERS] if N_USERS is not None else all_users
selected_user_set = set(selected_users)
top_item_set = set(top_items)

filtered = merged[
    merged["user_id"].isin(selected_user_set)
    & merged["product_id"].isin(top_item_set)
].copy()

user_history_raw = (
    filtered
    .groupby("user_id")["product_id"]
    .apply(list)
    .to_dict()
)

indexed_history = {}
MIN_HISTORY_LEN = 5

for user_id, items in user_history_raw.items():
    indexed_items = [item_to_index[item] for item in items if item in item_to_index]

    if len(indexed_items) >= MIN_HISTORY_LEN:
        indexed_history[user_id] = indexed_items

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

indexed_path = processed_dir / "indexed_history.pkl"

with open(indexed_path, "wb") as f:
    pickle.dump(indexed_history, f)

lengths = [len(v) for v in indexed_history.values()]
all_indexed_items = [item for hist in indexed_history.values() for item in hist]

print("\n===== Preprocessing Done =====")
print("Users selected:", len(selected_users))
print("Users after filtering:", len(indexed_history))
print("Top items kept:", len(top_items))
print("Total interactions:", sum(lengths))
print("Min history length:", min(lengths) if lengths else 0)
print("Max history length:", max(lengths) if lengths else 0)
print("Average history length:", sum(lengths) / len(lengths) if lengths else 0)
print("Unique indexed items:", len(set(all_indexed_items)))
print("Max indexed item id:", max(all_indexed_items) if all_indexed_items else None)
print("Saved:", indexed_path)
print("File size:", f"{indexed_path.stat().st_size / (1024 * 1024):.2f} MB")

Found raw files:
- orders: /content/drive/MyDrive/drl-product-recommendation/colab_inputs/orders.csv
- products: /content/drive/MyDrive/drl-product-recommendation/colab_inputs/products.csv
- prior: /content/drive/MyDrive/drl-product-recommendation/colab_inputs/order_products__prior.csv
- train: /content/drive/MyDrive/drl-product-recommendation/colab_inputs/order_products__train.csv

Raw shapes:
orders: (3421083, 7)
products: (49688, 4)
prior: (32434489, 4)
train: (1384617, 4)

===== Preprocessing Done =====
Users selected: 206209
Users after filtering: 197227
Top items kept: 1000
Total interactions: 18215667
Min history length: 5
Max history length: 2282
Average history length: 92.35889102404843
Unique indexed items: 1000
Max indexed item id: 999
Saved: data/processed/indexed_history.pkl
File size: 42.87 MB


## 6. Inspect + split train/val/test + infer `GLOBAL_ACTION_DIM`

Split theo thời gian:
- 70% đầu: train
- 15% giữa: validation
- 15% cuối: test

Dùng 15/15 thay vì 10/20 để validation không quá nhỏ.

In [ ]:
!python -m utils.inspect_data --data_path data/processed/indexed_history.pkl --state_size 5 --top_k 5
!python -m utils.split_history_train_val_test --input_path data/processed/indexed_history.pkl --train_output_path data/processed/train_history.pkl --val_output_path data/processed/val_history.pkl --test_output_path data/processed/test_history.pkl --train_ratio 0.7 --val_ratio 0.15 --test_ratio 0.15 --min_history_len 11
!python -m utils.inspect_data --data_path data/processed/train_history.pkl --state_size 5 --top_k 5
!python -m utils.inspect_data --data_path data/processed/val_history.pkl --state_size 5 --top_k 5
!python -m utils.inspect_data --data_path data/processed/test_history.pkl --state_size 5 --top_k 5

import pickle
with open("data/processed/indexed_history.pkl", "rb") as f:
    indexed_history = pickle.load(f)

all_items = [item for hist in indexed_history.values() for item in hist]
GLOBAL_ACTION_DIM = max(all_items) + 1

print("\n===== Global Action Space =====")
print("Max item id:", max(all_items))
print("Unique items:", len(set(all_items)))
print("GLOBAL_ACTION_DIM:", GLOBAL_ACTION_DIM)

## 7. Helper chạy lệnh

In [ ]:
import re
import subprocess
from pathlib import Path
import pandas as pd

Path("outputs/checkpoints").mkdir(parents=True, exist_ok=True)
Path("outputs/plots").mkdir(parents=True, exist_ok=True)
Path("outputs/logs").mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, title=None):
    if title:
        print("\n" + "=" * 100)
        print(title)
        print("=" * 100)

    print(cmd)

    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
    )

    if result.stdout:
        print("\nSTDOUT:")
        print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}: {cmd}")

    return result.stdout

## 8. Train DQN pure stable trên train

In [ ]:
run_cmd(
    "python -m training.train_dqn "
    "--data_path data/processed/train_history.pkl "
    f"--episodes {TRAIN_EPISODES} "
    f"--action_dim {GLOBAL_ACTION_DIM} "
    "--model_path outputs/checkpoints/dqn_pure_stable.pth "
    "--plot_dir outputs/plots "
    "--log_path outputs/logs/train_dqn_pure_stable.csv "
    "--gamma 0.9 "
    "--epsilon 1.0 "
    "--epsilon_min 0.05 "
    "--epsilon_decay 0.9995 "
    "--batch_size 64 "
    "--memory_size 50000 "
    "--lr 0.0001 "
    "--target_update_freq 200 "
    f"--top_k {TOP_K} "
    "--embedding_dim 32 "
    "--hidden_dim 128 "
    "--recent_boost 0.0 "
    f"--device {DEVICE}",
    title="Train DQN pure stable",
)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
Episode 6/5000 | Reward: -13.0 | Loss: 0.0000 | Epsilon: 1.0000
Episode 7/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000
Episode 8/5000 | Reward: -13.0 | Loss: 2.8200 | Epsilon: 0.9990
Episode 9/5000 | Reward: -16.0 | Loss: 2.7094 | Epsilon: 0.9980
Episode 10/5000 | Reward: -20.0 | Loss: 2.6561 | Epsilon: 0.9970
Episode 11/5000 | Reward: -20.0 | Loss: 2.4361 | Epsilon: 0.9960
Episode 12/5000 | Reward: -16.0 | Loss: 2.2870 | Epsilon: 0.9950
Episode 13/5000 | Reward: -20.0 | Loss: 1.9152 | Epsilon: 0.9940
Episode 14/5000 | Reward: -13.0 | Loss: 1.8892 | Epsilon: 0.9930
Episode 15/5000 | Reward: -20.0 | Loss: 1.6721 | Epsilon: 0.9920
Episode 16/5000 | Reward: -20.0 | Loss: 1.2005 | Epsilon: 0.9910
Episode 17/5000 | Reward: -20.0 | Loss: 0.9641 | Epsilon: 0.9900
Episode 18/5000 | Reward: -20.0 | Loss: 0.7991 | Epsilon: 0.9891
Episode 19/5000 | Reward: -20.0 | Loss: 0.8406 | Epsilon: 0.9881
Episode 20/5000 | Reward: -20.0 | Los

'Using action_dim: 1000\nUsing valid actions: 1000\nUsing embedding_dim: 32\nUsing hidden_dim: 128\nUsing recent_boost: 0.0\nEpisode 1/5000 | Reward: -12.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 2/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 3/5000 | Reward: -13.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 4/5000 | Reward: -13.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 5/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 6/5000 | Reward: -13.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 7/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 8/5000 | Reward: -13.0 | Loss: 2.8200 | Epsilon: 0.9990\nEpisode 9/5000 | Reward: -16.0 | Loss: 2.7094 | Epsilon: 0.9980\nEpisode 10/5000 | Reward: -20.0 | Loss: 2.6561 | Epsilon: 0.9970\nEpisode 11/5000 | Reward: -20.0 | Loss: 2.4361 | Epsilon: 0.9960\nEpisode 12/5000 | Reward: -16.0 | Loss: 2.2870 | Epsilon: 0.9950\nEpisode 13/5000 | Reward: -20.0 | Loss: 1.9152 | Epsilon: 0.9940\nEpisode 14/5000 | Reward: 

## 9. Train DQN + recency prior boost=2 trên train

In [ ]:
run_cmd(
    "python -m training.train_dqn "
    "--data_path data/processed/train_history.pkl "
    f"--episodes {TRAIN_EPISODES} "
    f"--action_dim {GLOBAL_ACTION_DIM} "
    "--model_path outputs/checkpoints/dqn_recency2_stable.pth "
    "--plot_dir outputs/plots "
    "--log_path outputs/logs/train_dqn_recency2_stable.csv "
    "--gamma 0.9 "
    "--epsilon 1.0 "
    "--epsilon_min 0.05 "
    "--epsilon_decay 0.9995 "
    "--batch_size 64 "
    "--memory_size 50000 "
    "--lr 0.0001 "
    "--target_update_freq 200 "
    f"--top_k {TOP_K} "
    "--embedding_dim 32 "
    "--hidden_dim 128 "
    "--recent_boost 2.0 "
    f"--device {DEVICE}",
    title="Train DQN + recency prior boost=2 stable",
)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
Episode 6/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000
Episode 7/5000 | Reward: -20.0 | Loss: 1.3516 | Epsilon: 0.9995
Episode 8/5000 | Reward: -20.0 | Loss: 2.6317 | Epsilon: 0.9985
Episode 9/5000 | Reward: -20.0 | Loss: 2.4889 | Epsilon: 0.9975
Episode 10/5000 | Reward: -2.0 | Loss: 0.0000 | Epsilon: 0.9975
Episode 11/5000 | Reward: -20.0 | Loss: 2.3945 | Epsilon: 0.9965
Episode 12/5000 | Reward: -20.0 | Loss: 2.2388 | Epsilon: 0.9955
Episode 13/5000 | Reward: -6.0 | Loss: 2.1459 | Epsilon: 0.9945
Episode 14/5000 | Reward: -20.0 | Loss: 1.8877 | Epsilon: 0.9935
Episode 15/5000 | Reward: -20.0 | Loss: 1.7260 | Epsilon: 0.9925
Episode 16/5000 | Reward: -20.0 | Loss: 1.4034 | Epsilon: 0.9915
Episode 17/5000 | Reward: -20.0 | Loss: 1.3433 | Epsilon: 0.9905
Episode 18/5000 | Reward: -13.0 | Loss: 0.9054 | Epsilon: 0.9896
Episode 19/5000 | Reward: -20.0 | Loss: 0.8234 | Epsilon: 0.9886
Episode 20/5000 | Reward: -20.0 | Loss:

'Using action_dim: 1000\nUsing valid actions: 1000\nUsing embedding_dim: 32\nUsing hidden_dim: 128\nUsing recent_boost: 2.0\nEpisode 1/5000 | Reward: -13.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 2/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 3/5000 | Reward: -13.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 4/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 5/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 6/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 7/5000 | Reward: -20.0 | Loss: 1.3516 | Epsilon: 0.9995\nEpisode 8/5000 | Reward: -20.0 | Loss: 2.6317 | Epsilon: 0.9985\nEpisode 9/5000 | Reward: -20.0 | Loss: 2.4889 | Epsilon: 0.9975\nEpisode 10/5000 | Reward: -2.0 | Loss: 0.0000 | Epsilon: 0.9975\nEpisode 11/5000 | Reward: -20.0 | Loss: 2.3945 | Epsilon: 0.9965\nEpisode 12/5000 | Reward: -20.0 | Loss: 2.2388 | Epsilon: 0.9955\nEpisode 13/5000 | Reward: -6.0 | Loss: 2.1459 | Epsilon: 0.9945\nEpisode 14/5000 | Reward: -2

## 10. Train DQN + recency prior boost=5 trên train

In [ ]:
run_cmd(
    "python -m training.train_dqn "
    "--data_path data/processed/train_history.pkl "
    f"--episodes {TRAIN_EPISODES} "
    f"--action_dim {GLOBAL_ACTION_DIM} "
    "--model_path outputs/checkpoints/dqn_recency5_stable.pth "
    "--plot_dir outputs/plots "
    "--log_path outputs/logs/train_dqn_recency5_stable.csv "
    "--gamma 0.9 "
    "--epsilon 1.0 "
    "--epsilon_min 0.05 "
    "--epsilon_decay 0.9995 "
    "--batch_size 64 "
    "--memory_size 50000 "
    "--lr 0.0001 "
    "--target_update_freq 200 "
    f"--top_k {TOP_K} "
    "--embedding_dim 32 "
    "--hidden_dim 128 "
    "--recent_boost 5.0 "
    f"--device {DEVICE}",
    title="Train DQN + recency prior boost=5 stable",
)

Streaming output truncated to the last 5000 lines.
Episode 6/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000
Episode 7/5000 | Reward: -20.0 | Loss: 1.3714 | Epsilon: 0.9995
Episode 8/5000 | Reward: -20.0 | Loss: 2.6588 | Epsilon: 0.9985
Episode 9/5000 | Reward: -13.0 | Loss: 2.6135 | Epsilon: 0.9975
Episode 10/5000 | Reward: -20.0 | Loss: 2.4760 | Epsilon: 0.9965
Episode 11/5000 | Reward: -20.0 | Loss: 2.3071 | Epsilon: 0.9955
Episode 12/5000 | Reward: -20.0 | Loss: 2.2057 | Epsilon: 0.9945
Episode 13/5000 | Reward: -20.0 | Loss: 1.8764 | Epsilon: 0.9935
Episode 14/5000 | Reward: -13.0 | Loss: 1.7015 | Epsilon: 0.9925
Episode 15/5000 | Reward: -20.0 | Loss: 1.4671 | Epsilon: 0.9915
Episode 16/5000 | Reward: -20.0 | Loss: 1.2451 | Epsilon: 0.9905
Episode 17/5000 | Reward: -20.0 | Loss: 0.9512 | Epsilon: 0.9896
Episode 18/5000 | Reward: -20.0 | Loss: 0.8101 | Epsilon: 0.9886
Episode 19/5000 | Reward: -20.0 | Loss: 0.6897 | Epsilon: 0.9876
Episode 20/5000 | Reward: -20.0 | Loss: 0.3

'Using action_dim: 1000\nUsing valid actions: 1000\nUsing embedding_dim: 32\nUsing hidden_dim: 128\nUsing recent_boost: 5.0\nEpisode 1/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 2/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 3/5000 | Reward: -16.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 4/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 5/5000 | Reward: -6.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 6/5000 | Reward: -20.0 | Loss: 0.0000 | Epsilon: 1.0000\nEpisode 7/5000 | Reward: -20.0 | Loss: 1.3714 | Epsilon: 0.9995\nEpisode 8/5000 | Reward: -20.0 | Loss: 2.6588 | Epsilon: 0.9985\nEpisode 9/5000 | Reward: -13.0 | Loss: 2.6135 | Epsilon: 0.9975\nEpisode 10/5000 | Reward: -20.0 | Loss: 2.4760 | Epsilon: 0.9965\nEpisode 11/5000 | Reward: -20.0 | Loss: 2.3071 | Epsilon: 0.9955\nEpisode 12/5000 | Reward: -20.0 | Loss: 2.2057 | Epsilon: 0.9945\nEpisode 13/5000 | Reward: -20.0 | Loss: 1.8764 | Epsilon: 0.9935\nEpisode 14/5000 | Reward: -

## 11. Chọn model bằng validation, rồi final test

Script `run_val_test_suite.py` sẽ:
1. Evaluate 3 DQN trên `val_history.pkl`.
2. Chọn model tốt nhất theo validation HitRate@5.
3. Evaluate Random / Popularity / Recent-item trên test.
4. Evaluate model đã chọn trên test.
5. Lưu CSV + Markdown report.

In [ ]:
run_cmd(
    "python -m evaluation.run_val_test_suite "
    "--val_data_path data/processed/val_history.pkl "
    "--test_data_path data/processed/test_history.pkl "
    f"--episodes {EVAL_EPISODES} "
    f"--top_k {TOP_K} "
    f"--action_dim {GLOBAL_ACTION_DIM} "
    "--embedding_dim 32 "
    "--hidden_dim 128 "
    f"--device {DEVICE} "
    "--checkpoint_dir outputs/checkpoints "
    "--pure_model dqn_pure_stable.pth "
    "--boost2_model dqn_recency2_stable.pth "
    "--boost5_model dqn_recency5_stable.pth "
    "--output_dir outputs/logs",
    title="Validation model selection and final test",
)

val_df = pd.read_csv("outputs/logs/validation_model_selection.csv")
test_df = pd.read_csv("outputs/logs/final_test_results.csv")

print("\nValidation model selection:")
display(val_df.sort_values("hit_rate_at_5", ascending=False))

print("\nFinal test results:")
display(test_df.sort_values("hit_rate_at_5", ascending=False))

## 12. Dong bo evaluation, ve bieu do va luu vao Drive

Cell nay dung dung `outputs/logs/final_test_results.csv` vua sinh ra de ve bieu do, luu value CSV/JSON/Markdown va anh PNG vao Google Drive.

<!-- AUTO_SYNC_EVALUATION_VISUALIZATION -->

In [ ]:
# AUTO_SYNC_EVALUATION_VISUALIZATION
from datetime import datetime
import json
import shutil
from pathlib import Path
from IPython.display import Image, display

required_metric_cols = [
    "hit_rate_at_5",
    "precision_at_5",
    "recall_at_5",
    "ndcg_at_5",
]

for name, df in [("validation_model_selection", val_df), ("final_test_results", test_df)]:
    missing = [col for col in required_metric_cols if col not in df.columns]
    if missing:
        raise ValueError(
            f"{name} thieu cot: {missing}. Hay chay lai notebook voi code evaluation moi."
        )

run_cmd(
    "python visualization/visualization.py "
    "--metrics_csv outputs/logs/final_test_results.csv "
    "--output_dir outputs/plots",
    title="Build synchronized evaluation visualizations",
)

recommendation_plot_path = Path("outputs/plots/recommendation_metrics.png")
training_plot_path = Path("outputs/plots/dqn_training.png")
for plot_path in [recommendation_plot_path, training_plot_path]:
    if not plot_path.exists():
        raise FileNotFoundError(f"Visualization image was not created: {plot_path}")

print("Recommendation metrics plot:", recommendation_plot_path)
display(Image(filename=str(recommendation_plot_path)))
print("DQN training plot:", training_plot_path)
display(Image(filename=str(training_plot_path)))

best_validation_row = val_df.sort_values("hit_rate_at_5", ascending=False).iloc[0].to_dict()
evaluation_values = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "metrics_csv": "outputs/logs/final_test_results.csv",
    "selected_by_validation": best_validation_row,
    "final_test_results": test_df.sort_values("hit_rate_at_5", ascending=False).to_dict(orient="records"),
    "validation_model_selection": val_df.sort_values("hit_rate_at_5", ascending=False).to_dict(orient="records"),
    "plots": [str(recommendation_plot_path), str(training_plot_path)],
}

evaluation_values_path = Path("outputs/logs/evaluation_values.json")
evaluation_values_path.write_text(
    json.dumps(evaluation_values, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved evaluation values JSON:", evaluation_values_path)

value_and_image_files = [
    Path("outputs/logs/final_test_results.csv"),
    Path("outputs/logs/validation_model_selection.csv"),
    Path("outputs/logs/train_val_test_report.md"),
    Path("outputs/logs/best_selected_by_validation.json"),
    Path("outputs/checkpoints/best_selected_by_validation.pth"),
    evaluation_values_path,
    recommendation_plot_path,
    training_plot_path,
]

if IN_COLAB:
    drive_eval_root = DRIVE_OUTPUT_DIR / "evaluation"
    drive_latest_dir = drive_eval_root / "latest"
    drive_timestamp_dir = drive_eval_root / datetime.now().strftime("%Y%m%d_%H%M%S")

    for target_dir in [drive_latest_dir, drive_timestamp_dir]:
        target_dir.mkdir(parents=True, exist_ok=True)
        for src in value_and_image_files:
            shutil.copy2(src, target_dir / src.name)

    print("Saved synchronized evaluation values and images to Drive:")
    print(drive_latest_dir)
    print("Timestamped backup:")
    print(drive_timestamp_dir)
else:
    print("Running locally. Evaluation values/images are saved under outputs/.")


## 13. Inspect model được chọn trên validation

In [ ]:
val_df = pd.read_csv("outputs/logs/validation_model_selection.csv")
best = val_df.sort_values("hit_rate_at_5", ascending=False).iloc[0]

BEST_MODEL_PATH = best["model_path"]
BEST_RECENT_BOOST = float(best["recent_boost"])

print("Best selected model:")
print(best)

run_cmd(
    "python -m evaluation.inspect_policy "
    "--data_path data/processed/test_history.pkl "
    f"--model_path {BEST_MODEL_PATH} "
    "--episodes 30 "
    f"--action_dim {GLOBAL_ACTION_DIM} "
    "--embedding_dim 32 "
    "--hidden_dim 128 "
    f"--recent_boost {BEST_RECENT_BOOST} "
    f"--device {DEVICE}",
    title="Inspect best selected model on test",
)

Best selected model:
split                                              validation
method                            DQN + recency prior boost=5
average_reward                                         -4.571
hit_rate_at_5                                          0.0368
episodes                                                 1000
model_path        outputs/checkpoints/dqn_recency5_stable.pth
recent_boost                                              5.0
type                                                      dqn
Name: 2, dtype: object

Inspect best selected model on test
python -m evaluation.inspect_policy --data_path data/processed/test_history.pkl --model_path outputs/checkpoints/dqn_recency5_stable.pth --episodes 30 --action_dim 1000 --embedding_dim 32 --hidden_dim 128 --recent_boost 5.0 --device cuda

STDOUT:

===== Policy Inspection =====
Using action_dim: 1000
Using valid actions: 1000
Using embedding_dim: 32
Using hidden_dim: 128
Using recent_boost: 5.0

--- Episode 1 ---
Step 1


'\n===== Policy Inspection =====\nUsing action_dim: 1000\nUsing valid actions: 1000\nUsing embedding_dim: 32\nUsing hidden_dim: 128\nUsing recent_boost: 5.0\n\n--- Episode 1 ---\nStep 1\n  State:   [np.int64(705), np.int64(533), np.int64(20), np.int64(44), np.int64(62)]\n  Target:  [53, 614, 290, 102, 705]\n  Actions: [533, 62, 705, 44, 20]\n  Hits:    1\n  Reward:  5.0\n\n--- Episode 2 ---\nStep 1\n  State:   [np.int64(3), np.int64(6), np.int64(160), np.int64(361), np.int64(834)]\n  Target:  [160, 861, 286, 821, 553]\n  Actions: [160, 0, 3, 6, 361]\n  Hits:    1\n  Reward:  5.0\nStep 2\n  State:   [np.int64(6), np.int64(160), np.int64(361), np.int64(834), np.int64(160)]\n  Target:  [861, 286, 821, 553, 160]\n  Actions: [834, 504, 908, 45, 305]\n  Hits:    0\n  Reward:  -2.0\nStep 3\n  State:   [np.int64(160), np.int64(361), np.int64(834), np.int64(160), np.int64(861)]\n  Target:  [286, 821, 553, 160, 33]\n  Actions: [861, 829, 144, 148, 492]\n  Hits:    0\n  Reward:  -2.0\n\n--- Episo

## 14. Build demo_data.json cho web demo va luu vao Drive

Cell nay chay `demo/build_demo_data.py` bang model tot nhat da chon o validation, tao `data/demo/demo_data.json`, copy vao `outputs/demo/` de zip kem, va luu mot ban vao Google Drive khi chay tren Colab.

<!-- AUTO_BUILD_DEMO_DATA_JSON -->

In [ ]:
# AUTO_BUILD_DEMO_DATA_JSON
from datetime import datetime
import shutil
from pathlib import Path

demo_output_path = Path("data/demo/demo_data.json")
demo_output_path.parent.mkdir(parents=True, exist_ok=True)

best_model_stem = Path(BEST_MODEL_PATH).stem
best_training_log = Path("outputs/logs") / f"train_{best_model_stem}.csv"
if not best_training_log.exists():
    print(f"Training log not found for best model: {best_training_log}")
    print("Fallback to outputs/logs/train_dqn_recency5_stable.csv")
    best_training_log = Path("outputs/logs/train_dqn_recency5_stable.csv")

run_cmd(
    "python demo/build_demo_data.py "
    "--train_history_path data/processed/train_history.pkl "
    "--eval_history_path data/processed/test_history.pkl "
    "--fallback_history_path data/processed/indexed_history.pkl "
    "--case_history_path data/processed/indexed_history.pkl "
    f"--model_path {BEST_MODEL_PATH} "
    "--metrics_csv outputs/logs/final_test_results.csv "
    f"--training_log {best_training_log} "
    f"--output_path {demo_output_path} "
    f"--recent_boost {BEST_RECENT_BOOST} "
    "--max_case_users 1000 "
    "--max_windows_per_user 20 "
    "--max_cases 300",
    title="Build web demo data",
)

if not demo_output_path.exists():
    raise FileNotFoundError(f"Demo data was not created: {demo_output_path}")

outputs_demo_dir = Path("outputs/demo")
outputs_demo_dir.mkdir(parents=True, exist_ok=True)
outputs_demo_path = outputs_demo_dir / "demo_data.json"
shutil.copy2(demo_output_path, outputs_demo_path)

print("Demo data created:", demo_output_path)
print(f"Demo data size: {demo_output_path.stat().st_size / (1024 * 1024):.2f} MB")
print("Copied into outputs for zip:", outputs_demo_path)

if IN_COLAB:
    drive_demo_dir = DRIVE_OUTPUT_DIR / "demo"
    drive_demo_dir.mkdir(parents=True, exist_ok=True)
    drive_demo_latest_path = drive_demo_dir / "demo_data.json"
    drive_demo_timestamped_path = drive_demo_dir / f"demo_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    shutil.copy2(demo_output_path, drive_demo_latest_path)
    shutil.copy2(demo_output_path, drive_demo_timestamped_path)

    print("Saved demo_data.json to Drive:")
    print(drive_demo_latest_path)
    print("Timestamped backup:")
    print(drive_demo_timestamped_path)

## 15. Zip outputs và copy về Google Drive

In [ ]:
from datetime import datetime
import shutil
from pathlib import Path
import os

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"train_val_test_outputs_{timestamp}"

if IN_COLAB:
    zip_dir = Path("/content")
    zip_path = zip_dir / zip_name
    
    print(f"Zipping outputs to {zip_path}.zip...")
    shutil.make_archive(str(zip_path), 'zip', 'outputs')
    
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    drive_zip_path = DRIVE_OUTPUT_DIR / f"{zip_name}.zip"
    shutil.copy2(f"{zip_path}.zip", drive_zip_path)
    
    print("\nSaved zip to Drive:")
    print(drive_zip_path)
    print(f"File size: {drive_zip_path.stat().st_size / (1024 * 1024):.2f} MB")
else:
    zip_dir = Path(os.getcwd())
    zip_path = zip_dir / zip_name
    print(f"Running locally. Zipping outputs to {zip_path}.zip...")
    shutil.make_archive(str(zip_path), 'zip', 'outputs')
    print("Done. Saved zip locally at:", f"{zip_path}.zip")

## Cách đọc kết quả

- Model được chọn bằng **validation HitRate@5**, không phải test.
- Test chỉ dùng cuối cùng để báo cáo.
- Nếu model được chọn dùng `recent_boost`, phải gọi là **DQN + recency prior**, không gọi là DQN thuần.
- Nếu Recent-item baseline vẫn cao nhất trên test, kết luận đúng là DQN hiện tại chưa thắng heuristic ngắn hạn mạnh nhất.